# Jev-CeNN — Global + Local Attention Repeated Twice: Faster 600-Step Selective 3→1 Lab

[![Run in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Jev_CeNN_GlobalLocal_x2_600Step_Colab.ipynb)

## Objective and motivation

The earlier selective W512 two-stage experiment reached only **0.96557 cosine** for group 0–2 at 450 distillation steps. Full-model recovery decreased agreement from **92.9167% at step 1** to **92.0833% at steps 20/40**, so adding more of the same costly end-to-end training is unlikely to help without altering the architecture and selection.

**New architecture, per original three-layer group:**

\`\`\`text
Teacher:       [Global Attention + FFN] -> [Sliding Attention + FFN] -> [Sliding Attention + FFN]
New candidate: [FULL attention] -> [sliding W129] -> [CeNN + FFN]
            -> [FULL attention] -> [sliding W129] -> [CeNN + FFN]
\`\`\`

Exactly **four sequential attention passes** with a **three-stage curriculum**. The candidate first learns with four global attention passes, then converts the second pass to sliding W129, and finally converts the fourth pass to sliding W129. No QKV, CeNN or FFN parameters are discarded or reinitialized during conversion. The final deployable topology is global/local/global/local, with two multiscale cellular residuals, two feedforward networks, and an optional rank-64 correction. This entire candidate occupies ONE ModernBERT encoder slot; the two subsequent native teacher blocks are full-layer bypasses *only when accepted*. **Global really means all valid tokens** (not a centered W512 mask that fails to reach the far edge). The new block does MORE attention work than the three original layers; any GPU speedup is a hypothesis, not an assumption.

## 600 optimizer updates maximum per candidate

- Stage A: **520 local feature-distillation optimizer updates** split into **100 full/full/full/full, 140 full/W129/full/full, and 280 full/W129/full/W129** updates. Teacher triple inputs/targets and actual *current student-prefix states* are cached once per group. **Only the final architecture's PROBE scores select checkpoint weights.** Unlike the previous Colab, each local update does **NOT** run the frozen 28-layer teacher and student encoder.
- Stage B: **80 end-to-end optimizer updates** through the evolving student, training ONLY the candidate; prior accepted groups, remaining native layers and task head are frozen. Early stopping selects a DEV-GATE checkpoint and prevents damage from recovery.
- **TOTAL 600 updates per attempted group**, including recovery; no hidden CeNN pretraining updates. CeNN gains and weights learn during both stages.
- Use native SDPA for all four attention stages, no sparse Python K/V gather, FP32 by default for stability. Training features are cached only for the *current* group and released before training the next candidate.

## Scientifically selective

Each candidate is evaluated against the original teacher and the **already compressed current student**. Retain candidate only if **real student-prefix local cosine ≥0.99**, NMSE ≤0.025, cumulative teacher decision agreement ≥0.99 and gold drop ≤1.5 percentage points. Otherwise restore the original three full encoder blocks. Optionally require measured GPU speedup.

Data: case-disjoint TRAIN (700), DEV-PROBE (80), DEV-GATE (140), final sealed HOLDOUT (200), all sourced from official dataset TRAIN. Original benchmark TEST is not accessed. The final HOLDOUT is evaluated **once after all accept/reject decisions**, not used for tuning. Since teacher pretrained on source benchmark TRAIN, this holdout is independent only of the present experiment, not necessarily teacher pretraining.

The notebook saves a complete JSON/CSV report, candidate checkpoints, selective acceptance reasons, paired B1/B2 short/long CUDA timings and an audit figure. **No automatic model publishing or claims of 0.99 convergence before GPU execution.**

## Correction after the first run

The prior log reaches cosine **0.99150** for layers 0–2 and **0.99652** for layers 3–5, yet still loses teacher agreement (97.5% and 98.75% at the best recorded recovery points). This is a **downstream decision fidelity** problem, not simply lack of local convergence. This curriculum is a new hypothesis: it may improve optimization stability, but global warmup itself can cost *more* GPU time. The strict 99% task agreement gate is unchanged. Recovery checkpoint selection now refuses any candidate that reduces agreement from the pre-recovery checkpoint.


In [ ]:
#@title 1. Install pinned teacher and latest GlobalLocal×2 CeNN dependencies
%pip -q install "transformers==4.57.6" "datasets>=2.20,<5" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
%pip -q install -U "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@main"


In [ ]:
#@title 2. Four attention passes, strict 600-update budget and original teacher
from pathlib import Path
import copy, json, math, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from transformers.models.modernbert.modeling_modernbert import apply_rotary_pos_emb
from tinycenn_lm.laya_lab.pdelta import PDelta3GDN2CLVRAttention
from tinycenn_lm.laya_lab.core import _depthwise_sequence_conv, _valid_tokens
from tinycenn_lm.standalone_decision import (
    QTYPES, build_checkpoint_model, build_sequence, collate_items,
    normalize_question,
)
from tinycenn_lm.decision_training import require_finite, decision_distill_loss, guarded_step
from tinycenn_lm.sliding_window_fusion import SlidingWindowFusion

SEED=2028
SOURCE_MODEL="convaiinnovations/laya-typed-decisions" #@param {type:"string"}
MAX_LEN=512
FULL_ATTENTION_CONTEXT=512 #@param {type:"integer"}
LOCAL_WINDOW=129 #@param {type:"integer"}
CORRECTION_RANK=64 #@param {type:"integer"}
TRAIN_CASES=700 #@param {type:"integer"}
DEV_CASES=220 #@param {type:"integer"}
HOLDOUT_CASES=200 #@param {type:"integer"}
PROBE_CASES=80 #@param {type:"integer"}
BATCH=2 #@param {type:"integer"}
PROBE_BATCHES=12 #@param {type:"integer"}
DEV_GATE_DECISIONS=240 #@param {type:"integer"}
HOLDOUT_DECISIONS=500 #@param {type:"integer"}
TOTAL_OPTIMIZER_STEPS=600 #@param {type:"integer"}
LOCAL_DISTILL_STEPS=520 #@param {type:"integer"}
FULL_ATTENTION_STEPS=100 #@param {type:"integer"}
FIRST_SLIDING_STEPS=140 #@param {type:"integer"}
BOTH_SLIDING_STEPS=280 #@param {type:"integer"}
FULL_MODEL_RECOVERY_STEPS=80 #@param {type:"integer"}
CACHE_BATCHES=96 #@param {type:"integer"}
PROBE_EVERY=65 #@param {type:"integer"}
FEEDBACK_EVERY=20 #@param {type:"integer"}
COSINE_WEIGHT=3.0
PAIR_INTERMEDIATE_WEIGHT=0.20
INTERMEDIATE_WEIGHT=0.10
CENN_AUX_WEIGHT=0.02
GROUP_LR=0.00015
QKV_LR=0.000015
FEEDBACK_LR=0.000004 #@param {type:"number"}
HIDDEN_ALIGN_WEIGHT=0.05 #@param {type:"number"}
TARGET_COSINE=0.99
TARGET_NMSE=0.025
TARGET_AGREEMENT=0.99
MAX_GOLD_DROP=0.015
MAX_INCREMENTAL_AGREEMENT_DROP=0.01
REQUIRE_CANDIDATE_SPEEDUP=False #@param {type:"boolean"}
USE_FP16_AMP=False #@param {type:"boolean"}
BENCH_REPEATS=10 #@param {type:"integer"}
OUTPUT_DIR=Path("/content/Jev_CeNN_GlobalLocal_x2_600Step")
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
assert MAX_LEN==512 and FULL_ATTENTION_CONTEXT==512 and BATCH>0
assert LOCAL_WINDOW==129
assert TOTAL_OPTIMIZER_STEPS==LOCAL_DISTILL_STEPS+FULL_MODEL_RECOVERY_STEPS==600
assert FULL_ATTENTION_STEPS+FIRST_SLIDING_STEPS+BOTH_SLIDING_STEPS==LOCAL_DISTILL_STEPS
assert FULL_ATTENTION_STEPS>0 and FIRST_SLIDING_STEPS>0 and BOTH_SLIDING_STEPS>0
RECOVERY_MAX_AGREEMENT_DROP=0.0 # select recovery only without reducing agreement
assert CACHE_BATCHES>=PROBE_BATCHES
assert TRAIN_CASES>0 and DEV_CASES>PROBE_CASES>0 and HOLDOUT_CASES>0
assert 0<HOLDOUT_DECISIONS and 0<DEV_GATE_DECISIONS
assert PROBE_BATCHES>=2 and LOCAL_DISTILL_STEPS>=1
assert 0<=FULL_MODEL_RECOVERY_STEPS
assert PROBE_EVERY>=1 and FEEDBACK_EVERY>=1

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("This notebook needs a CUDA GPU (Colab > Runtime > Change runtime type).")
torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda")
amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16

source_dir = Path(snapshot_download(
    SOURCE_MODEL,
    allow_patterns=["rl_agent_config.json", "model.safetensors", "encoder/*", "tokenizer/*"],
))
teacher, source_cfg = build_checkpoint_model(source_dir)
student_eval, _ = build_checkpoint_model(source_dir)
tokenizer = AutoTokenizer.from_pretrained(str(source_dir / "tokenizer"))
teacher.to(device).eval().requires_grad_(False)
student_eval.to(device).eval().requires_grad_(False)
teacher.encoder.config.reference_compile = False
student_eval.encoder.config.reference_compile = False

print("GPU:", torch.cuda.get_device_name(0), "| AMP:", amp_dtype)
from tinycenn_lm.standalone_decision import full_attention_indices
GLOBAL_LAYERS = full_attention_indices(teacher)
SLIDING_LAYERS = [i for i in range(len(teacher.encoder.layers)) if i not in GLOBAL_LAYERS]
assert len(teacher.encoder.layers)==28 and GLOBAL_LAYERS==list(range(0,28,3))
for i in GLOBAL_LAYERS:
    assert tuple(teacher.encoder.layers[i].attn.local_attention) == (-1, -1)
print("Global layers:", GLOBAL_LAYERS, "| native sliding:", SLIDING_LAYERS)
print("Four attention passes: FULL -> W",LOCAL_WINDOW," -> FULL -> W",LOCAL_WINDOW,"| total optimizer updates:",TOTAL_OPTIMIZER_STEPS)
assert all(tuple(teacher.encoder.layers[i].attn.local_attention)==(64,64) for i in SLIDING_LAYERS)
print("Selective groups; all rejected groups stay native. Final HOLDOUT is sealed.")
print("Teacher:", SOURCE_MODEL, "| official test split stays untouched")


In [ ]:
#@title 3. Partition dataset by CASE: TRAIN, PROBE, DEV-GATE and sealed HOLDOUT
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon_labels.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer,
                state,
                q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

from tinycenn_lm.jev_cenn_selective import (
    verify_case_partition, judge_replacement, plan_active_layers,
)
rows=list(load_dataset("LocalLLaMA/typed-decisions","all",split="train"))
rng=random.Random(SEED)
rng.shuffle(rows)
assert TRAIN_CASES+DEV_CASES+HOLDOUT_CASES<=len(rows), (
    f"Required TRAIN+DEV+HOLDOUT cases exceeds {len(rows)} dataset TRAIN cases")
train_rows=rows[:TRAIN_CASES]
dev_rows=rows[TRAIN_CASES:TRAIN_CASES+DEV_CASES]
holdout_rows=rows[TRAIN_CASES+DEV_CASES:TRAIN_CASES+DEV_CASES+HOLDOUT_CASES]
split_info=verify_case_partition(train_rows,dev_rows,holdout_rows)
probe_rows=dev_rows[:PROBE_CASES]
gate_rows=dev_rows[PROBE_CASES:]
verify_case_partition(train_rows,probe_rows,gate_rows)
assert not {str(x["id"]) for x in holdout_rows}&{
    str(x["id"]) for x in dev_rows}
train_items=make_items_from_rows(train_rows,SEED,True)
probe_items=make_items_from_rows(probe_rows,SEED+1,True)
gate_items=make_items_from_rows(gate_rows,SEED+2,True)
# IMPORTANT: holdout_items are NOT constructed yet. The final evaluation
# cell materializes these previously sealed rows after all decisions are made.
assert len(gate_items)>=DEV_GATE_DECISIONS
assert len(probe_items)>=PROBE_BATCHES*BATCH
print("Independent partitions:",split_info,
      "probe cases",len(probe_rows),"gate cases",len(gate_rows))
print("Optimizer/probe/gate decisions:",len(train_items),
      len(probe_items),len(gate_items))

def direct_batch(selected):
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k: v.to(device) for k, v in b.items()}

def batch(items, offset, n=BATCH):
    selected = [items[(offset + i) % len(items)] for i in range(n)]
    return direct_batch(selected)

# Only case-isolated TRAIN / PROBE / DEV-GATE inputs are available here.
# The independent HOLDOUT remains sealed until the final evaluation cell.
print("TRAIN / PROBE / DEV-GATE decisions:",
      len(train_items), len(probe_items), len(gate_items))
print("Official TEST untouched; independent HOLDOUT not materialized.")


In [ ]:
#@title 4. Enumerate the nine prospective 3→1 groups (none forced)
from tinycenn_lm.jev_cenn_full_encoder import (
    group_plan, preserved_tail, install_group, restore_group
)
from tinycenn_lm.jev_cenn_full_local_repeat import (
    FullLocalRepeatCeNNSuperLayer, sdpa_full_or_window
)
from tinycenn_lm.jev_cenn_selective import (
    judge_replacement, plan_active_layers
)
GROUPS=group_plan(len(teacher.encoder.layers))
assert GROUPS==tuple((i,i+1,i+2) for i in range(0,27,3))
assert preserved_tail(28)==(27,)
assert all(g[0] in GLOBAL_LAYERS for g in GROUPS)
assert all(g[1] in SLIDING_LAYERS and g[2] in SLIDING_LAYERS for g in GROUPS)
assert LOCAL_WINDOW==129
print("Candidate triples:",GROUPS)
print("Curriculum: 0 FULL/FULL/FULL/FULL for",FULL_ATTENTION_STEPS,"updates")
print("Curriculum: 1 FULL/W129/FULL/FULL for",FIRST_SLIDING_STEPS,"updates")
print("Curriculum: 2 FULL/W129/FULL/W129 for",BOTH_SLIDING_STEPS,"updates")
assert sum((FULL_ATTENTION_STEPS,FIRST_SLIDING_STEPS,BOTH_SLIDING_STEPS))==LOCAL_DISTILL_STEPS
print("Budget per triple:",LOCAL_DISTILL_STEPS,"local +",
      FULL_MODEL_RECOVERY_STEPS,"changed-model =",TOTAL_OPTIMIZER_STEPS)


In [ ]:
#@title 5. Cache teacher triplets + CURRENT student-prefix activations ONCE per candidate
import gc, csv
def cpu_snapshot(model):
    return {k:v.detach().to("cpu").clone() for k,v in model.state_dict().items()}
def check_finite(tensor,caption):
    if not torch.isfinite(tensor.float()).all():
        raise FloatingPointError(f"Nonfinite {caption}")
def sample_distinct(items,n,seed):
    rng=random.Random(seed)
    chosen=rng.sample(items,min(len(items),n))
    return [chosen[i:i+BATCH] for i in range(0,len(chosen)-BATCH+1,BATCH)]
def capture_group_example(start,selected):
    inputs=direct_batch(selected)
    found={}
    def pre(mod,args,kwargs):
        found["x"]=args[0].detach()
        found["position_ids"]=kwargs.get("position_ids")
    def post_first(mod,args,kwargs,result):
        found["pair_target"]=result[0].detach()
    def post_last(mod,args,kwargs,result):
        found["end_target"]=result[0].detach()
    hooks=[
        teacher.encoder.layers[start].register_forward_pre_hook(pre,with_kwargs=True),
        teacher.encoder.layers[start+1].register_forward_hook(post_first,with_kwargs=True),
        teacher.encoder.layers[start+2].register_forward_hook(post_last,with_kwargs=True),
    ]
    try:
        with torch.inference_mode(),torch.autocast(
                "cuda",dtype=amp_dtype,enabled=USE_FP16_AMP):
            teacher(**inputs)
    finally:
        for h in hooks:h.remove()
    current={}
    def pre_student(mod,args,kwargs):
        current["x"]=args[0].detach()
    hook=student_eval.encoder.layers[start].register_forward_pre_hook(
        pre_student,with_kwargs=True)
    try:
        with torch.inference_mode(),torch.autocast(
                "cuda",dtype=amp_dtype,enabled=USE_FP16_AMP):
            student_eval(**inputs)
    finally:
        hook.remove()
    if not all(k in found for k in ("x","position_ids","pair_target","end_target")):
        raise RuntimeError("ModernBERT teacher group hook incomplete")
    if "x" not in current:
        raise RuntimeError("ModernBERT compressed-prefix hook incomplete")
    data={
        "x":found["x"],"rollout":current["x"],
        "pair_target":found["pair_target"],
        "end_target":found["end_target"],
        "valid":inputs["attention_mask"].bool(),
        "position_ids":found["position_ids"],
    }
    for k,v in data.items():
        if torch.is_tensor(v) and v.is_floating_point():
            check_finite(v,f"{k} group {start}")
    # CPU FP32 retains numerical stability; release all GPU cached states.
    return {k:(v.detach().float().cpu().contiguous()
               if torch.is_tensor(v) and v.is_floating_point()
               else v.detach().cpu().contiguous() if torch.is_tensor(v)
               else v) for k,v in data.items()}
def to_cuda(row):
    return {k:v.to(device,non_blocking=True) if torch.is_tensor(v) else v
            for k,v in row.items()}
def prepare_group_cache(start):
    train_batches=sample_distinct(
        train_items,CACHE_BATCHES*BATCH,SEED+start*79+1)
    probe_batches=[
        probe_items[i*BATCH:(i+1)*BATCH] for i in range(PROBE_BATCHES)]
    train_cache=[]
    probe_cache=[]
    for label,batches,dest in (
        ("TRAIN",train_batches,train_cache),
        ("PROBE",probe_batches,probe_cache)):
        for i,chosen in enumerate(batches):
            dest.append(capture_group_example(start,chosen))
            if (i+1)%32==0 or i+1==len(batches):
                print(f"  {start}-{start+2} {label} cached "
                      f"{i+1}/{len(batches)} batches")
    return train_cache,probe_cache
print("Caching strategy: frozen teacher/current prefix ONE TIME per group,",
      CACHE_BATCHES*BATCH,"diverse TRAIN decisions.")


In [ ]:
#@title 6. Cosine/NMSE and cached independent DEV-GATE teacher predictions
def feature_loss(output,reference,valid):
    pred=output.float()[valid]
    gold=reference.float()[valid]
    # Source activations are validated during cache construction; the
    # differentiable scalar loss is checked once per update. Avoid CUDA
    # sync in these two per-forward tensor checks.
    nmse=(pred-gold).square().sum()/gold.square().sum().clamp_min(1e-8)
    cosine=F.cosine_similarity(pred,gold,dim=-1).mean()
    # Do not call Python float() inside the 520-step CUDA training loop.
    # Converting GPU tensors to floats forces a device synchronization.
    return nmse+COSINE_WEIGHT*(1-cosine),nmse.detach(),cosine.detach()
@torch.inference_mode()
def cached_local_quality(candidate,cache,key="rollout"):
    candidate.eval()
    cs=err=energy=tokens=0.0
    for row in cache:
        e=to_cuda(row)
        with torch.autocast("cuda",dtype=amp_dtype,enabled=USE_FP16_AMP):
            pred=candidate(e[key],attention_mask=e["valid"],
                           position_ids=e["position_ids"])[0]
        a=pred.float()[e["valid"]]
        b=e["end_target"].float()[e["valid"]]
        check_finite(a,"local prediction")
        cs+=float(F.cosine_similarity(a,b,dim=-1).sum())
        err+=float((a-b).square().sum())
        energy+=float(b.square().sum())
        tokens+=int(e["valid"].sum())
    return {"cosine":cs/max(tokens,1),"nmse":err/max(energy,1e-8),
            "tokens":int(tokens)}
def cached_rank(teacher_view,prefix_view):
    return (0.20*(teacher_view["nmse"]+COSINE_WEIGHT*(1-teacher_view["cosine"]))
           +0.80*(prefix_view["nmse"]+COSINE_WEIGHT*(1-prefix_view["cosine"])))
def cache_teacher_predictions(items):
    examples=[]
    for i in range(0,len(items),BATCH):
        inp=direct_batch(items[i:i+BATCH])
        with torch.inference_mode(),torch.autocast(
                "cuda",dtype=amp_dtype,enabled=USE_FP16_AMP):
            logits,_=teacher(**inp)
        examples.append({
            "batch":inp,
            "teacher":logits.float().detach(),
            "gold":torch.tensor(
                [r["gold_index"] for r in items[i:i+BATCH]],
                device=device,dtype=torch.long)})
    return examples
GATE_ITEMS=gate_items[:DEV_GATE_DECISIONS]
assert len(GATE_ITEMS)==DEV_GATE_DECISIONS
GATE_CACHE=cache_teacher_predictions(GATE_ITEMS)
@torch.inference_mode()
def decision_quality(model,examples):
    model.eval()
    correct=student_gold=teacher_gold=n=0
    js_sum=0.
    for e in examples:
        with torch.autocast("cuda",dtype=amp_dtype,enabled=USE_FP16_AMP):
            logits,_=model(**e["batch"])
        check_finite(logits,"full-model decision logits")
        mask=e["batch"]["marker_mask"].bool()
        a=torch.softmax(e["teacher"].masked_fill(~mask,-1e4),-1)
        b=torch.softmax(logits.float().masked_fill(~mask,-1e4),-1)
        av,bv=a.argmax(-1),b.argmax(-1)
        center=(a+b)/2
        js_sum+=float((0.5*(a*(a.clamp_min(1e-9).log()-
                                  center.clamp_min(1e-9).log())).sum(-1)
                      +0.5*(b*(b.clamp_min(1e-9).log()-
                                  center.clamp_min(1e-9).log())).sum(-1)).sum())
        correct+=int((av==bv).sum())
        student_gold+=int((bv==e["gold"]).sum())
        teacher_gold+=int((av==e["gold"]).sum())
        n+=len(av)
    return {"decisions":n,"agreement":correct/max(n,1),
            "student_gold_accuracy":student_gold/max(n,1),
            "teacher_gold_accuracy":teacher_gold/max(n,1),
            "mean_js":js_sum/max(n,1)}
ORIGINAL_GATE=decision_quality(student_eval,GATE_CACHE)
assert ORIGINAL_GATE["agreement"]==1.
print("Original (no compression) DEV-GATE:",ORIGINAL_GATE)


In [ ]:
#@title 7. Exactly 520 fast cached feature-distillation optimizer steps per candidate
def train_cached_group(block,train_cache,probe_cache,start):
    block.requires_grad_(True)
    qkv=[];other=[]
    for name,p in block.named_parameters():
        (qkv if ".qkv." in name else other).append(p)
    optimizer=torch.optim.AdamW([
        {"params":qkv,"lr":QKV_LR,"base":QKV_LR},
        {"params":other,"lr":GROUP_LR,"base":GROUP_LR},
    ],weight_decay=2e-4)
    best_teacher=cached_local_quality(block,probe_cache,"x")
    best_rollout=cached_local_quality(block,probe_cache,"rollout")
    best_score=cached_rank(best_teacher,best_rollout)
    best_step=0
    best_weights=cpu_snapshot(block)
    history=[{"step":0,"teacher":best_teacher,"real_prefix":best_rollout}]
    indices=list(range(len(train_cache)))
    rng=random.Random(SEED+start*177)
    index=0
    for step in range(1,LOCAL_DISTILL_STEPS+1):
        if index==0:
            rng.shuffle(indices)
        row=to_cuda(train_cache[indices[index]])
        index=(index+1)%len(indices)
        optimizer.zero_grad(set_to_none=True)
        block.train()
        # Train primarily on ACTUAL prefix input; teacher view provides
        # intermittent anchoring without doubling attention compute.
        entry=row["x"] if step%5==0 else row["rollout"]
        with torch.autocast("cuda",dtype=amp_dtype,enabled=USE_FP16_AMP):
            prediction=block(
                entry,attention_mask=row["valid"],
                position_ids=row["position_ids"])[0]
            loss,_,_=feature_loss(
                prediction,row["end_target"],row["valid"])
            if block.last_first is not None:
                intermediate,_,_=feature_loss(
                    block.last_first,row["pair_target"],row["valid"])
                loss=loss+PAIR_INTERMEDIATE_WEIGHT*intermediate
        require_finite(loss,f"four-attention group {start} distillation")
        loss.backward()
        torch.nn.utils.clip_grad_norm_(block.parameters(),0.6)
        lr_multiplier=0.12+0.88*0.5*(1+math.cos(math.pi*step/LOCAL_DISTILL_STEPS))
        for group in optimizer.param_groups:
            group["lr"]=group["base"]*lr_multiplier
        optimizer.step()
        if step==1 or step%PROBE_EVERY==0 or step==LOCAL_DISTILL_STEPS:
            a=cached_local_quality(block,probe_cache,"x")
            b=cached_local_quality(block,probe_cache,"rollout")
            rank=cached_rank(a,b)
            history.append({"step":step,"teacher":a,"real_prefix":b})
            if rank<best_score:
                best_score,best_step=rank,step
                best_weights=cpu_snapshot(block)
            print(f"   group {start}-{start+2} step {step:3d}/"
                  f"{LOCAL_DISTILL_STEPS}: teacher={a['cosine']:.5f}, "
                  f"REAL prefix={b['cosine']:.5f}, best={best_step}")
    block.load_state_dict(best_weights,strict=True)
    block.eval().requires_grad_(False)
    del optimizer,best_weights
    return {"total_optimizer_steps":LOCAL_DISTILL_STEPS,
            "best_step":best_step,"probe_history":history,
            "best_real_prefix":cached_local_quality(
                block,probe_cache,"rollout")}


In [ ]:
#@title 8. At most 80 full-changed-student recovery updates (stop on regression)
def quality_rank(q):
    return (3.0*(1-q["agreement"])+
            1.2*(1-q["student_gold_accuracy"])+2*q["mean_js"])
def task_recover(block,start):
    assert block.curriculum_phase==2, "Never recover a non-final full-attention curriculum"
    initial=decision_quality(student_eval,GATE_CACHE)
    if FULL_MODEL_RECOVERY_STEPS==0:
        return {"updates":0,"selected_step":0,"initial":initial,
                "final":initial,"history":[]}
    # Only THIS candidate gets gradients; all prior accepted groups and the
    # original native remainder/head remain frozen and in eval() mode.
    block.requires_grad_(True)
    student_eval.eval()
    optimizer=torch.optim.AdamW(block.parameters(),lr=FEEDBACK_LR,
                                 weight_decay=0.0001)
    index=random.Random(SEED+start*331).sample(
        list(range(len(train_items))),min(len(train_items),
                                          FULL_MODEL_RECOVERY_STEPS*BATCH))
    best_rank=quality_rank(initial)
    best_step=0
    best_weights=cpu_snapshot(block)
    history=[{"step":0,"quality":initial}]
    stale=0
    actual_steps=0
    for step in range(1,FULL_MODEL_RECOVERY_STEPS+1):
        candidates=[train_items[index[((step-1)*BATCH+j)%len(index)]]
                    for j in range(BATCH)]
        inp=direct_batch(candidates)
        gold=torch.tensor([z["gold_index"] for z in candidates],
                          device=device,dtype=torch.long)
        optimizer.zero_grad(set_to_none=True)
        captured={}
        def teacher_h(mod,args,out):
            captured["teacher"]=out.last_hidden_state.detach()
        def student_h(mod,args,out):
            captured["student"]=out.last_hidden_state
        th=teacher.encoder.register_forward_hook(teacher_h)
        sh=student_eval.encoder.register_forward_hook(student_h)
        try:
            with torch.no_grad(),torch.autocast(
                    "cuda",dtype=amp_dtype,enabled=USE_FP16_AMP):
                teacher_logits,teacher_action=teacher(**inp)
            with torch.autocast("cuda",dtype=amp_dtype,enabled=USE_FP16_AMP):
                student_logits,student_action=student_eval(**inp)
                kd=decision_distill_loss(
                    student_logits,teacher_logits,
                    inp["marker_mask"].bool(),
                    student_action,teacher_action)
                supervised=F.cross_entropy(student_logits.float(),gold)
                hidden,_,_=feature_loss(
                    captured["student"],captured["teacher"],
                    inp["attention_mask"].bool())
                loss=kd+0.2*supervised+HIDDEN_ALIGN_WEIGHT*hidden
            require_finite(loss,"changed-student 3-to-1 recovery")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(block.parameters(),0.4)
            optimizer.step()
        finally:
            th.remove()
            sh.remove()
            captured.clear()
        actual_steps=step
        if step==1 or step%FEEDBACK_EVERY==0 or step==FULL_MODEL_RECOVERY_STEPS:
            observed=decision_quality(student_eval,GATE_CACHE)
            history.append({"step":step,"quality":observed})
            rank=quality_rank(observed)
            # Preserve teacher agreement: the previous run selected
            # recovery updates that improved JS but harmed decisions.
            if (rank<best_rank
                and observed["agreement"]>=
                    initial["agreement"]-RECOVERY_MAX_AGREEMENT_DROP
                and observed["student_gold_accuracy"]>=
                    initial["student_gold_accuracy"]-MAX_GOLD_DROP):
                best_rank,best_step=rank,step
                best_weights=cpu_snapshot(block)
                stale=0
            else:
                stale+=1
            print(f"  recovery {start}-{start+2} step "
                  f"{step}/{FULL_MODEL_RECOVERY_STEPS}: "
                  f"agreement={observed['agreement']:.4%}, "
                  f"gold={observed['student_gold_accuracy']:.4%}, "
                  f"selected={best_step}")
            if stale>=3:
                print("  Early stop: full-model recovery does not help")
                break
    block.load_state_dict(best_weights,strict=True)
    block.eval().requires_grad_(False)
    final=decision_quality(student_eval,GATE_CACHE)
    del optimizer,best_weights
    return {"updates":actual_steps,"selected_step":best_step,
            "initial":initial,"final":final,"history":history,
            "best_step_zero_means_full_model_recovery_hurt":best_step==0}
@torch.inference_mode()
def cuda_latency(model,inp,repeats=BENCH_REPEATS):
    model.eval()
    with torch.autocast("cuda",dtype=amp_dtype,enabled=USE_FP16_AMP):
        for _ in range(3):model(**inp)
        torch.cuda.synchronize()
        durations=[]
        for _ in range(repeats):
            a=torch.cuda.Event(enable_timing=True)
            b=torch.cuda.Event(enable_timing=True)
            a.record()
            model(**inp)
            b.record()
            b.synchronize()
            durations.append(float(a.elapsed_time(b)))
    return {"median_ms":float(np.median(durations)),
            "p95_ms":float(np.percentile(durations,95)),
            "repeats":repeats}


In [ ]:
#@title 9. Try nine original triples; accept or restore with strict cumulative quality
ACCEPTED=[]
TRIALS=[]
speed_sample=GATE_CACHE[0]["batch"]
for start,middle,end in GROUPS:
    print("\n"+"="*81)
    print("3→1 candidate",start,middle,end,"| existing selected:",ACCEPTED)
    print("="*81)
    current=decision_quality(student_eval,GATE_CACHE)
    cache,probe=prepare_group_cache(start)
    candidate=FullLocalRepeatCeNNSuperLayer(
        teacher.encoder.layers[start],
        teacher.encoder.layers[middle],
        teacher.encoder.layers[end],
        window=LOCAL_WINDOW,max_len=MAX_LEN,
        correction_rank=CORRECTION_RANK).to(device)
    distill=train_cached_group(candidate,cache,probe,start)
    assert candidate.curriculum_phase==2
    assert candidate.active_attention_topology==(
        "full",f"window{LOCAL_WINDOW}","full",f"window{LOCAL_WINDOW}")
    # Preserve ORIGINAL group modules for complete rollback.
    originals=install_group(student_eval,(start,middle,end),candidate)
    recovery=task_recover(candidate,start)
    quality=decision_quality(student_eval,GATE_CACHE)
    local_real=cached_local_quality(candidate,probe,"rollout")
    local_teacher=cached_local_quality(candidate,probe,"x")
    candidate.set_cenn_enabled(False)
    local_without_cenn=cached_local_quality(candidate,probe,"rollout")
    candidate.set_cenn_enabled(True)
    with torch.inference_mode():
        # Compared on the SAME DEV-GATE batch and with consistent CUDA events.
        elapsed_candidate=cuda_latency(student_eval,speed_sample)
    restore_group(student_eval,(start,middle,end),originals)
    elapsed_native=cuda_latency(student_eval,speed_sample)
    speedup=elapsed_native["median_ms"]/max(
        elapsed_candidate["median_ms"],1e-9)
    verdict=judge_replacement(
        current,quality,local_cosine=local_real["cosine"],
        local_nmse=local_real["nmse"],
        min_agreement=TARGET_AGREEMENT,
        max_incremental_agreement_drop=MAX_INCREMENTAL_AGREEMENT_DROP,
        max_gold_drop=MAX_GOLD_DROP,min_local_cosine=TARGET_COSINE,
        max_local_nmse=TARGET_NMSE,speedup=speedup,
        require_speedup=REQUIRE_CANDIDATE_SPEEDUP)
    if verdict["accepted"]:
        # Reinstall the frozen selected checkpoint, not the last optimizer step.
        install_group(student_eval,(start,middle,end),candidate)
        ACCEPTED.append((start,middle,end))
        file=OUTPUT_DIR/f"accepted_full_local_x2_{start:02d}_{end:02d}.pt"
        torch.save({
            "architecture":candidate.architecture,
            "indices":[start,middle,end],
            "attention_stages":["full","window","full","window"],
            "curriculum_phase":candidate.curriculum_phase,
            "curriculum_steps":[FULL_ATTENTION_STEPS,
                                FIRST_SLIDING_STEPS,BOTH_SLIDING_STEPS],
            "window":LOCAL_WINDOW,
            "state":cpu_snapshot(candidate),
            "selection":verdict,
        },file)
        result="accepted"
        print("ACCEPTED",start,end,"-> active blocks",28-2*len(ACCEPTED))
    else:
        file=None
        result="rejected_rolled_back"
        restored=decision_quality(student_eval,GATE_CACHE)
        assert abs(restored["agreement"]-current["agreement"])<1e-9
        print("REJECTED",start,end,":",verdict["reasons"])
        del candidate
    TRIALS.append({
        "group":[start,middle,end],"status":result,
        "feature_distillation":distill,
        "full_model_recovery":recovery,
        "real_prefix_fidelity":local_real,
        "teacher_input_fidelity":local_teacher,
        "cenn_heldout_probe_cosine_gain":local_real["cosine"]-
                                        local_without_cenn["cosine"],
        "quality_before":current,"quality_candidate":quality,
        "gate":verdict,"speedup":speedup,
        "candidate_gpu":elapsed_candidate,
        "native_gpu":elapsed_native,
        "total_optimizer_updates_executed":
            LOCAL_DISTILL_STEPS+recovery["updates"],
        "checkpoint":str(file) if file else None,
    })
    assert TRIALS[-1]["total_optimizer_updates_executed"]<=600
    (OUTPUT_DIR/"full_local_x2_progress.json").write_text(
        json.dumps({
            "accepted_groups":[list(g) for g in ACCEPTED],
            "trials":TRIALS,
            "current_DEV_GATE":decision_quality(student_eval,GATE_CACHE)
        },indent=2,allow_nan=False))
    del cache,probe,originals
    gc.collect()
    torch.cuda.empty_cache()
FINAL_STRUCTURE=plan_active_layers(28,ACCEPTED)
FINAL_DEV=decision_quality(student_eval,GATE_CACHE)
print("SELECTED active encoder blocks:",FINAL_STRUCTURE)
print("FINAL DEV-GATE teacher agreement:",FINAL_DEV["agreement"])


In [ ]:
#@title 10. ONE-TIME independent HOLDOUT evaluation (not used to train/select)
HOLDOUT_ITEMS=make_items_from_rows(holdout_rows,SEED+3,True)
assert len(HOLDOUT_ITEMS)>=HOLDOUT_DECISIONS
HOLDOUT_ITEMS=HOLDOUT_ITEMS[:HOLDOUT_DECISIONS]
HOLDOUT_CACHE=cache_teacher_predictions(HOLDOUT_ITEMS)
FINAL_HOLDOUT=decision_quality(student_eval,HOLDOUT_CACHE)
ORIGINAL_HOLDOUT=decision_quality(teacher,HOLDOUT_CACHE)
assert ORIGINAL_HOLDOUT["agreement"]==1.0
@torch.inference_mode()
def encoder_holdout_fidelity(items,n_batches=8):
    cosine=diff=norm=ntokens=0.
    for i in range(min(n_batches,math.ceil(len(items)/BATCH))):
        inp=direct_batch(items[i*BATCH:(i+1)*BATCH])
        with torch.autocast("cuda",dtype=amp_dtype,enabled=USE_FP16_AMP):
            t=teacher.encoder(input_ids=inp["input_ids"],
                attention_mask=inp["attention_mask"]).last_hidden_state
            s=student_eval.encoder(input_ids=inp["input_ids"],
                attention_mask=inp["attention_mask"]).last_hidden_state
        mask=inp["attention_mask"].bool()
        a,b=s.float()[mask],t.float()[mask]
        check_finite(a,"HOLDOUT final student representation")
        check_finite(b,"HOLDOUT frozen teacher representation")
        cosine+=float(F.cosine_similarity(a,b,dim=-1).sum())
        diff+=float((a-b).square().sum())
        norm+=float(b.square().sum())
        ntokens+=int(mask.sum())
    return {"cosine":cosine/max(ntokens,1),
            "nmse":diff/max(norm,1e-8),"tokens":int(ntokens)}
HOLDOUT_ENCODER=encoder_holdout_fidelity(HOLDOUT_ITEMS)
print("Sealed HOLDOUT decisions:",FINAL_HOLDOUT)
print("Sealed HOLDOUT representation:",HOLDOUT_ENCODER)


In [ ]:
#@title 11. Actual paired GPU speed: B1/B2, short/long, same padded tensors
@torch.inference_mode()
def paired_cuda(teacher,student,inp,repeats=BENCH_REPEATS):
    teacher.eval();student.eval()
    measurements={"teacher":[],"student":[]}
    with torch.autocast("cuda",dtype=amp_dtype,enabled=USE_FP16_AMP):
        for _ in range(4):
            teacher(**inp);student(**inp)
        torch.cuda.synchronize()
        for trial in range(repeats):
            ordered=(("teacher",teacher),("student",student)) if trial%2==0 else (
                ("student",student),("teacher",teacher))
            for label,model in ordered:
                before=torch.cuda.Event(enable_timing=True)
                after=torch.cuda.Event(enable_timing=True)
                before.record()
                model(**inp)
                after.record()
                after.synchronize()
                measurements[label].append(float(before.elapsed_time(after)))
    return {
        "teacher_median_ms":float(np.median(measurements["teacher"])),
        "student_median_ms":float(np.median(measurements["student"])),
        "teacher_p95_ms":float(np.percentile(measurements["teacher"],95)),
        "student_p95_ms":float(np.percentile(measurements["student"],95)),
        "speedup":float(np.median(measurements["teacher"]))/max(
            float(np.median(measurements["student"])),1e-9),
        "raw_ms":measurements,
        "batch":int(inp["input_ids"].shape[0]),
        "padded_tokens":int(inp["input_ids"].shape[1]),
    }
ordered=sorted(HOLDOUT_ITEMS,key=lambda x:len(x["ids"]))
shapes=[
    ("short_B1",ordered[:1]),
    ("long_B1",ordered[-1:]),
    ("short_B2",ordered[:2]),
    ("long_B2",ordered[-2:]),
]
BENCHMARKS=[]
for label,sample in shapes:
    m=paired_cuda(teacher,student_eval,direct_batch(sample))
    m["label"]=label
    BENCHMARKS.append(m)
    print(f"{label}: teacher {m['teacher_median_ms']:.2f}ms "
          f"student {m['student_median_ms']:.2f}ms "
          f"speedup {m['speedup']:.3f}x")
PARAM_COUNT={
    "teacher":sum(p.numel() for p in teacher.parameters()),
    "selected_student":sum(p.numel() for p in student_eval.parameters()),
}
PARAM_COUNT["reduction_fraction"]=1-PARAM_COUNT["selected_student"]/PARAM_COUNT["teacher"]
print("Parameter count:",PARAM_COUNT)


In [ ]:
#@title 12. Write complete JSON/CSV/report figure and experimental checkpoint
import matplotlib.pyplot as plt
holdout_pass=(
    FINAL_HOLDOUT["agreement"]>=TARGET_AGREEMENT and
    FINAL_HOLDOUT["student_gold_accuracy"]>=
        FINAL_HOLDOUT["teacher_gold_accuracy"]-MAX_GOLD_DROP)
speed_pass=all(z["speedup"]>1 for z in BENCHMARKS)
ACCEPTED_FOR_DEPLOY=bool(ACCEPTED and holdout_pass and speed_pass)
STATUS=("EXPERIMENTAL_GATE_PASSED" if ACCEPTED_FOR_DEPLOY
        else "EXPERIMENTAL_NOT_DEPLOYABLE")
REPORT={
    "experiment":"jev_cenn_full_global_local_global_local_3to1_600updates",
    "source_model":SOURCE_MODEL,
    "attention_sequence":["global","window129","global","window129"],
    "max_attention_context":MAX_LEN,
    "native_sdpa_four_passes":True,
    "cenn_module_count_per_superlayer":2,
    "ffn_count_per_superlayer":2,
    "total_optimizer_budget_per_candidate":TOTAL_OPTIMIZER_STEPS,
    "curriculum":"full_first_then_convert_sliding_one_at_a_time",
    "curriculum_steps":{
        "four_global":FULL_ATTENTION_STEPS,
        "first_local_only":FIRST_SLIDING_STEPS,
        "both_local_final":BOTH_SLIDING_STEPS,
    },
    "no_attention_weight_reset_at_transition":True,
    "checkpoint_selection_final_topology_only":True,
    "local_distillation_budget":LOCAL_DISTILL_STEPS,
    "full_model_recovery_budget":FULL_MODEL_RECOVERY_STEPS,
    "cached_train_batches_per_candidate":CACHE_BATCHES,
    "case_disjoint_data":{"train_cases":TRAIN_CASES,"probe_cases":PROBE_CASES,
                          "gate_cases":DEV_CASES-PROBE_CASES,
                          "holdout_cases":HOLDOUT_CASES},
    "official_test_used":False,"selection_used_holdout":False,
    "source_teacher_training_data_independence_unverified":True,
    "group_trials":TRIALS,"accepted_groups":[list(g) for g in ACCEPTED],
    "selected_structure":FINAL_STRUCTURE,
    "final_development_quality":FINAL_DEV,
    "independent_one_time_holdout_quality":FINAL_HOLDOUT,
    "independent_one_time_holdout_encoder":HOLDOUT_ENCODER,
    "cuda_paired_benchmarks":BENCHMARKS,
    "parameter_count":PARAM_COUNT,
    "holdout_quality_pass":bool(holdout_pass),
    "speed_pass_all_shapes":bool(speed_pass),
    "deployable_candidate":ACCEPTED_FOR_DEPLOY,"status":STATUS,
}
report_file=OUTPUT_DIR/"jev_cenn_global_local_x2_600_report.json"
report_file.write_text(json.dumps(REPORT,indent=2,allow_nan=False))
table=OUTPUT_DIR/"jev_cenn_global_local_x2_600_trials.csv"
with table.open("w",newline="") as out:
    names=["group","status","real_prefix_cosine","real_prefix_nmse",
           "teacher_agreement","student_gold","speedup","reason","optimizer_updates"]
    writer=csv.DictWriter(out,fieldnames=names)
    writer.writeheader()
    for x in TRIALS:
        writer.writerow({
            "group":"-".join(map(str,x["group"])),
            "status":x["status"],
            "real_prefix_cosine":x["real_prefix_fidelity"]["cosine"],
            "real_prefix_nmse":x["real_prefix_fidelity"]["nmse"],
            "teacher_agreement":x["quality_candidate"]["agreement"],
            "student_gold":x["quality_candidate"]["student_gold_accuracy"],
            "speedup":x["speedup"],
            "reason":";".join(x["gate"]["reasons"]),
            "optimizer_updates":x["total_optimizer_updates_executed"],
        })
model_file=OUTPUT_DIR/f"jev_cenn_global_local_x2_600_{STATUS}.pt"
torch.save({
    "architecture":"jev_cenn_global_local_x2_600",
    "attention_sequence":["global","window129","global","window129"],
    "groups":[list(g) for g in ACCEPTED],
    "states":{
        str(g[0]):cpu_snapshot(student_eval.encoder.layers[g[0]])
        for g in ACCEPTED},
    "report":REPORT,
},model_file)
fig,ax=plt.subplots(figsize=(10,5))
xs=np.arange(len(TRIALS))
ax.bar(xs,[x["quality_candidate"]["agreement"]*100 for x in TRIALS],
       color=["tab:green" if x["status"]=="accepted" else "tab:red"
              for x in TRIALS])
ax.axhline(TARGET_AGREEMENT*100,linestyle="--",color="gray")
ax.set_xticks(xs)
ax.set_xticklabels(["-".join(map(str,x["group"])) for x in TRIALS],
                   rotation=40)
ax.set(xlabel="Candidate original teacher layers",
       ylabel="Full student DEV teacher agreement (%)",
       title="Global→Window→Global→Window CeNN: candidate accept/reject")
fig.tight_layout()
figure=OUTPUT_DIR/"jev_cenn_global_local_x2_600_plot.png"
fig.savefig(figure,dpi=160,bbox_inches="tight")
plt.show()
print("FINAL STATUS:",STATUS)
print("Accepted triples:",ACCEPTED,"| active blocks:",FINAL_STRUCTURE["active_encoder_blocks"])
print("HOLDOUT:",FINAL_HOLDOUT)
print("Files:",report_file,table,model_file,figure)
print("Hugging Face model unchanged; official benchmark TEST untouched.")
